In [3]:
import pandas as pd
from datetime import timedelta
# Load data from CSV
df = pd.read_csv('common/MachineLearningModel/output/five_mins/EURUSD_5_Min_testing.csv')

df['datetime'] = pd.to_datetime(df['datetime'])
df['UTC'] = df['datetime'] + timedelta(hours=5)

# Calculate GMT (UTC + 2 hours)
df['GMT'] = df['UTC'] + timedelta(hours=2)

# Define constants
MINOR_MIN_EXTREME_HEIGHT_ATRS = 2.0
MAJOR_TO_MINOR_HEIGHT_RATIO = 2.5
MINOR_MIN_EXTREME_WIDTH = 2
MAJOR_MIN_EXTREME_WIDTH = 2
RANGE_AVERAGING_PERIOD = 250
# LINE_VALUE_DOWN = -1.0
# LINE_VALUE_UP = 1.0
LINE_MINOR = 'minor'
LINE_MAJOR = 'major'
LINE_SHADOW = 'shadow'
LINE_STABLE = 'stable'
NoRepaint = False
LINE_VALUE_UP = 1.0
LINE_VALUE_FLAT = 0.0
LINE_VALUE_DOWN = -1.0
# Initialize columns for signals and other intermediate calculations
df['ATR'] = df['high'] - df['low']
df['ATR_SMA'] = df['ATR'].rolling(window=RANGE_AVERAGING_PERIOD).mean()
df['MinorMinExtremeHeight'] = df['ATR_SMA'] * MINOR_MIN_EXTREME_HEIGHT_ATRS
df['MajorMinExtremeHeight'] = df['MinorMinExtremeHeight'] * MAJOR_TO_MINOR_HEIGHT_RATIO
df['line1'] = 0.0
df['line2'] = 0.0
df['line3'] = 0.0
df['line4'] = 0.0
df['line5'] = 0.0

# Initialize state variables
minor_low_extreme_price = df['low'].iloc[0]
minor_hi_extreme_price = df['high'].iloc[0]
major_low_extreme_price = df['low'].iloc[0]
major_hi_extreme_price = df['high'].iloc[0]
minor_low_extreme_idx = 0
minor_hi_extreme_idx = 0
major_low_extreme_idx = 0
major_hi_extreme_idx = 0
minor_extreme_mode = 0
major_extreme_mode = 0
first_minor_low = True
first_minor_high = True
first_major_low = True
first_major_high = True

def eraseExtreme(lineType, barIdx, value):
    drawShadow = (lineType == LINE_MAJOR) and (NoRepaint or (value == LINE_VALUE_UP and df['line1'].iloc[barIdx] != 0) or (value == LINE_VALUE_DOWN and df['line2'].iloc[barIdx] != 0))
    drawExtreme(lineType, barIdx, LINE_VALUE_FLAT)
    if drawShadow:
        draw(LINE_SHADOW, barIdx, value)

def drawExtreme(lineType, barIdx, value):
    if not NoRepaint:
        draw(lineType, barIdx, value)
        drawStableLine(lineType, barIdx, value)

def drawStableLine(lineType, barIdx, value):
    if lineType == LINE_MAJOR:
        return False
    draw(LINE_STABLE, barIdx, value)
    return True

def draw(lineType, barIdx, value):
    if lineType == LINE_MAJOR:
        updateLine('line1', 'line2', barIdx, value)
    elif lineType == LINE_MINOR:
        updateLine('line5', 'line5', barIdx, value)
    elif lineType == LINE_SHADOW:
        updateLine('line3', 'line3', barIdx, value)
    elif lineType == LINE_STABLE:
        updateLine('line4', 'line4', barIdx, value)

def updateLine(lineUp, lineDown, barIdx, value):
    if value in [LINE_VALUE_FLAT, LINE_VALUE_UP]:
        df.loc[barIdx, lineUp] = value
    if value in [LINE_VALUE_FLAT, LINE_VALUE_DOWN]:
        df.loc[barIdx, lineDown] = value

# Helper functions
def check_for_extremes(low_extreme_idx, low_extreme_price, hi_extreme_idx, hi_extreme_price, first_low, first_high, extreme_mode, min_extreme_height, min_extreme_width, current_idx, low, high, lineType, df):
    signal = 0
    line_value = 0.0
    # Check for Bottom
    if extreme_mode > -1:
        if low < low_extreme_price:
            if not first_low:
                eraseExtreme(lineType, low_extreme_idx, LINE_VALUE_DOWN)
            low_extreme_price = low
            low_extreme_idx = current_idx
            first_low = False
        elif low > low_extreme_price:
            drawExtreme(lineType, low_extreme_idx, LINE_VALUE_DOWN)
            first_low = False
            if ((low - low_extreme_price) >= min_extreme_height) and ((current_idx - low_extreme_idx) >= min_extreme_width):
                extreme_mode = -1
                hi_extreme_price = high
                hi_extreme_idx = current_idx
                first_high = True
                first_low = True
                line_value = LINE_VALUE_DOWN
                if NoRepaint:
                    draw(lineType, low_extreme_idx, LINE_VALUE_DOWN)
                drawStableLine(lineType, low_extreme_idx, LINE_VALUE_FLAT)
                # if signal_type == 'minor':
                #     signal = 1  # Minor buy signal
                #     df.at[low_extreme_idx, 'line1'] = line_value
                # else:
                #     signal = 2  # Major buy signal
                #     df.at[low_extreme_idx, 'line2'] = line_value

    # Check for Top
    if extreme_mode < 1:
        if high > hi_extreme_price:
            if not first_high:
                eraseExtreme(lineType, hi_extreme_idx, LINE_VALUE_UP)
            hi_extreme_price = high
            hi_extreme_idx = current_idx
            first_high = False
        elif high < hi_extreme_price:
            drawExtreme(lineType, hi_extreme_idx, LINE_VALUE_UP)
            first_high = False
            if ((hi_extreme_price - low) >= min_extreme_height) and ((current_idx - hi_extreme_idx) >= min_extreme_width):
                extreme_mode = 1
                low_extreme_price = low
                low_extreme_idx = current_idx
                first_high = True
                first_low = True
                line_value = LINE_VALUE_UP
                if NoRepaint:
                    draw(lineType, hi_extreme_idx, LINE_VALUE_UP)
                drawStableLine(lineType, hi_extreme_idx, LINE_VALUE_FLAT)
                # if signal_type == 'minor':
                #     signal = -1  # Minor sell signal
                #     df.at[hi_extreme_idx, 'line1'] = line_value
                # else:
                #     signal = -2  # Major sell signal
                #     df.at[hi_extreme_idx, 'line2'] = line_value

    return low_extreme_idx, hi_extreme_idx, low_extreme_price, hi_extreme_price, first_low, first_high, extreme_mode, signal

# Process each row
for idx in range(1, len(df)):
    # Minor extremes

    minor_low_extreme_idx, minor_hi_extreme_idx, minor_low_extreme_price, minor_hi_extreme_price, first_minor_low, first_minor_high, minor_extreme_mode, minor_signal = check_for_extremes(
        minor_low_extreme_idx, minor_low_extreme_price,
        minor_hi_extreme_idx, minor_hi_extreme_price, first_minor_low, first_minor_high, minor_extreme_mode,
        df['MinorMinExtremeHeight'].iloc[idx], MINOR_MIN_EXTREME_WIDTH, idx, df['low'].iloc[idx], df['high'].iloc[idx], 'minor', df
    )

    # Major extremes

    major_low_extreme_idx, major_hi_extreme_idx, major_low_extreme_price, major_hi_extreme_price, first_major_low, first_major_high, major_extreme_mode, major_signal = check_for_extremes(
        major_low_extreme_idx, major_low_extreme_price,
        major_hi_extreme_idx, major_hi_extreme_price, first_major_low, first_major_high, major_extreme_mode,
        df['MajorMinExtremeHeight'].iloc[idx], MAJOR_MIN_EXTREME_WIDTH, idx, df['low'].iloc[idx], df['high'].iloc[idx], 'major', df
    )

# Save the result to a new CSV
df.to_csv("common/MachineLearningModel/output/test_result_9.csv", index=False)


In [2]:
import pandas as pd
import numpy as np

# Constants
ALERT_TOP = 1
ALERT_BOTTOM = 2
ALERT_STABLE = 3

LINE_MINOR = 1
LINE_MAJOR = 2
LINE_SHADOW = 3
LINE_STABLE = 4

LINE_VALUE_UP = 1.0
LINE_VALUE_FLAT = 0.0
LINE_VALUE_DOWN = -1.0

MODE_LOOKING_FOR_BOTTOM = 1
MODE_LOOKING_FOR_TOP = -1
MODE_UNDETERMINED = 0

RANGE_AVERAGING_PERIOD = 250

# Read CSV file
df = pd.read_csv('common/MachineLearningModel/output/five_mins/EURUSD_5_Min_testing.csv')

# Initialize variables
df['line1'] = np.nan
df['line2'] = np.nan
df['line3'] = np.nan
df['line4'] = np.nan
df['line5'] = np.nan

# Initialize dictionaries
minor_dict = {
    'LowExtremeIdx': 1,
    'FirstLow': True,
    'HiExtremeIdx': 1,
    'FirstHigh': True,
    'ExtremeMode': 0,
    'LowExtremePrice': None,
    'HiExtremePrice': None
}

major_dict = {
    'LowExtremeIdx': 1,
    'FirstLow': True,
    'HiExtremeIdx': 1,
    'FirstHigh': True,
    'ExtremeMode': 0,
    'LowExtremePrice': None,
    'HiExtremePrice': None
}


NoRepaint = False
MinorMinExtremeHeightATRs = 2.0
MajorToMinorHeightRatio = 2.5
MinorMinExtremeWidth = 2
MajorMinExtremeWidth = 2

def average(df, index):
    if index < RANGE_AVERAGING_PERIOD:
        return df['high'].rolling(window=RANGE_AVERAGING_PERIOD).mean().iloc[index]
    else:
        return df['high'].rolling(window=RANGE_AVERAGING_PERIOD).mean().iloc[index]

def processBar(index):
    if index == 0:
        return
    elif index == 1:
        minor_dict['LowExtremePrice'] = getLow(index)
        minor_dict['HiExtremePrice'] = getHigh(index)
        major_dict['LowExtremePrice'] = getLow(index)
        major_dict['HiExtremePrice'] = getHigh(index)
    else:
        MinorMinExtremeHeight = average(df, index) * MinorMinExtremeHeightATRs
        MajorMinExtremeHeight = MinorMinExtremeHeight * MajorToMinorHeightRatio
        checkForExtremes(index, minor_dict, MinorMinExtremeHeight, MinorMinExtremeWidth, LINE_MINOR)
        checkForExtremes(index, major_dict, MajorMinExtremeHeight, MajorMinExtremeWidth, LINE_MAJOR)

def checkForExtremes(index, extreme_dict, MinExtremeHeight, MinExtremeWidth, lineType):
    global NoRepaint

    if extreme_dict['ExtremeMode'] > -1:
        if getLow(index) < extreme_dict['LowExtremePrice']:
            if not extreme_dict['FirstLow']:
                eraseExtreme(lineType, extreme_dict['LowExtremeIdx'], LINE_VALUE_DOWN)
            extreme_dict['LowExtremePrice'] = getLow(index)
            extreme_dict['LowExtremeIdx'] = index
            extreme_dict['FirstLow'] = False
        elif getLow(index) > extreme_dict['LowExtremePrice']:
            drawExtreme(lineType, extreme_dict['LowExtremeIdx'], LINE_VALUE_DOWN)
            extreme_dict['FirstLow'] = False

            if ((getLow(index) - extreme_dict['LowExtremePrice']) >= MinExtremeHeight) and ((index - extreme_dict['LowExtremeIdx']) >= MinExtremeWidth):
                extreme_dict['ExtremeMode'] = -1
                extreme_dict['HiExtremePrice'] = getHigh(index)
                extreme_dict['HiExtremeIdx'] = index
                extreme_dict['FirstHigh'] = True
                extreme_dict['FirstLow'] = True

                if NoRepaint:
                    draw(lineType, extreme_dict['LowExtremeIdx'], LINE_VALUE_DOWN)
                drawStableLine(lineType, extreme_dict['LowExtremeIdx'], LINE_VALUE_FLAT)

    if extreme_dict['ExtremeMode'] < 1:
        if getHigh(index) > extreme_dict['HiExtremePrice']:
            if not extreme_dict['FirstHigh']:
                eraseExtreme(lineType, extreme_dict['HiExtremeIdx'], LINE_VALUE_UP)
            extreme_dict['HiExtremePrice'] = getHigh(index)
            extreme_dict['HiExtremeIdx'] = index
            extreme_dict['FirstHigh'] = False
        elif getHigh(index) < extreme_dict['HiExtremePrice']:
            drawExtreme(lineType, extreme_dict['HiExtremeIdx'], LINE_VALUE_UP)
            extreme_dict['FirstHigh'] = False

            if ((extreme_dict['HiExtremePrice'] - getHigh(index)) >= MinExtremeHeight) and ((index - extreme_dict['HiExtremeIdx']) >= MinExtremeWidth):
                extreme_dict['ExtremeMode'] = 1
                extreme_dict['LowExtremePrice'] = getLow(index)
                extreme_dict['LowExtremeIdx'] = index
                extreme_dict['FirstHigh'] = True
                extreme_dict['FirstLow'] = True

                if NoRepaint:
                    draw(lineType, extreme_dict['HiExtremeIdx'], LINE_VALUE_UP)
                drawStableLine(lineType, extreme_dict['HiExtremeIdx'], LINE_VALUE_FLAT)

    draw(lineType, index, LINE_VALUE_FLAT)

def eraseExtreme(lineType, barIdx, value):
    drawShadow = (lineType == LINE_MAJOR) and (NoRepaint or (value == LINE_VALUE_UP and df['line1'].iloc[barIdx] != 0) or (value == LINE_VALUE_DOWN and df['line2'].iloc[barIdx] != 0))
    drawExtreme(lineType, barIdx, LINE_VALUE_FLAT)
    if drawShadow:
        draw(LINE_SHADOW, barIdx, value)

def drawExtreme(lineType, barIdx, value):
    if not NoRepaint:
        draw(lineType, barIdx, value)
        drawStableLine(lineType, barIdx, value)

def drawStableLine(lineType, barIdx, value):
    if lineType == LINE_MAJOR:
        return False
    draw(LINE_STABLE, barIdx, value)
    return True

def draw(lineType, barIdx, value):
    if lineType == LINE_MAJOR:
        updateLine('line1', 'line2', barIdx, value)
    elif lineType == LINE_MINOR:
        updateLine('line5', 'line5', barIdx, value)
    elif lineType == LINE_SHADOW:
        updateLine('line3', 'line3', barIdx, value)
    elif lineType == LINE_STABLE:
        updateLine('line4', 'line4', barIdx, value)

def updateLine(lineUp, lineDown, barIdx, value):
    if value in [LINE_VALUE_FLAT, LINE_VALUE_UP]:
        df.loc[barIdx, lineUp] = value
    if value in [LINE_VALUE_FLAT, LINE_VALUE_DOWN]:
        df.loc[barIdx, lineDown] = value

def getLow(index):
    return df['low'].iloc[index]

def getHigh(index):
    return df['high'].iloc[index]

# Process all bars
for idx in range(len(df)):
    processBar(idx)

# Save the results to a new CSV file
df.to_csv("common/MachineLearningModel/output/test_result_8.csv", index=False)


In [1]:
import pandas as pd
from datetime import timedelta

# Load data from CSV
df = pd.read_csv('common/MachineLearningModel/output/five_mins/EURUSD_5_Min_testing.csv')

df['datetime'] = pd.to_datetime(df['datetime'])
df['UTC'] = df['datetime'] + timedelta(hours=5)
df['GMT'] = df['UTC'] + timedelta(hours=2)

# Define constants
MINOR_MIN_EXTREME_HEIGHT_ATRS = 2.0
MAJOR_TO_MINOR_HEIGHT_RATIO = 2.5
MINOR_MIN_EXTREME_WIDTH = 2
MAJOR_MIN_EXTREME_WIDTH = 2
RANGE_AVERAGING_PERIOD = 250
LINE_VALUE_UP = 1.0
LINE_VALUE_FLAT = 0.0
LINE_VALUE_DOWN = -1.0
LINE_MINOR = 'minor'
LINE_MAJOR = 'major'
LINE_SHADOW = 'shadow'
LINE_STABLE = 'stable'
NoRepaint = False

# Initialize columns for signals and other intermediate calculations
df['ATR'] = df['high'] - df['low']
df['ATR_SMA'] = df['ATR'].rolling(window=RANGE_AVERAGING_PERIOD).mean()
df['MinorMinExtremeHeight'] = df['ATR_SMA'] * MINOR_MIN_EXTREME_HEIGHT_ATRS
df['MajorMinExtremeHeight'] = df['MinorMinExtremeHeight'] * MAJOR_TO_MINOR_HEIGHT_RATIO
df[['line1', 'line2', 'line3', 'line4', 'line5']] = 0.0

# Initialize state variables
state_vars = {
    'minor': {
        'low_extreme_price': df['low'].iloc[0], 'hi_extreme_price': df['high'].iloc[0],
        'low_extreme_idx': 0, 'hi_extreme_idx': 0,
        'extreme_mode': 0, 'first_low': True, 'first_high': True
    },
    'major': {
        'low_extreme_price': df['low'].iloc[0], 'hi_extreme_price': df['high'].iloc[0],
        'low_extreme_idx': 0, 'hi_extreme_idx': 0,
        'extreme_mode': 0, 'first_low': True, 'first_high': True
    }
}

def eraseExtreme(lineType, barIdx, value):
    drawShadow = (lineType == LINE_MAJOR) and (NoRepaint or (value == LINE_VALUE_UP and df.loc[barIdx, 'line1'] != 0) or (value == LINE_VALUE_DOWN and df.loc[barIdx, 'line2'] != 0))
    drawExtreme(lineType, barIdx, LINE_VALUE_FLAT)
    if drawShadow:
        draw(LINE_SHADOW, barIdx, value)

def drawExtreme(lineType, barIdx, value):
    if not NoRepaint:
        draw(lineType, barIdx, value)
        drawStableLine(lineType, barIdx, value)

def drawStableLine(lineType, barIdx, value):
    if lineType == LINE_MAJOR:
        return False
    draw(LINE_STABLE, barIdx, value)
    return True

def draw(lineType, barIdx, value):
    if lineType == LINE_MAJOR:
        updateLine('line1', 'line2', barIdx, value)
    elif lineType == LINE_MINOR:
        updateLine('line5', 'line5', barIdx, value)
    elif lineType == LINE_SHADOW:
        updateLine('line3', 'line3', barIdx, value)
    elif lineType == LINE_STABLE:
        updateLine('line4', 'line4', barIdx, value)

def updateLine(lineUp, lineDown, barIdx, value):
    if value in [LINE_VALUE_FLAT, LINE_VALUE_UP]:
        df.loc[barIdx, lineUp] = value
    if value in [LINE_VALUE_FLAT, LINE_VALUE_DOWN]:
        df.loc[barIdx, lineDown] = value

def check_for_extremes(low_extreme_idx, low_extreme_price, hi_extreme_idx, hi_extreme_price, first_low, first_high, extreme_mode, min_extreme_height, min_extreme_width, current_idx, low, high, lineType):
    signal = 0
    line_value = 0.0
    # Check for Bottom
    if extreme_mode > -1:
        if low < low_extreme_price:
            if not first_low:
                eraseExtreme(lineType, low_extreme_idx, LINE_VALUE_DOWN)
            low_extreme_price = low
            low_extreme_idx = current_idx
            first_low = False
        elif low > low_extreme_price:
            drawExtreme(lineType, low_extreme_idx, LINE_VALUE_DOWN)
            first_low = False
            if ((low - low_extreme_price) >= min_extreme_height) and ((current_idx - low_extreme_idx) >= min_extreme_width):
                extreme_mode = -1
                hi_extreme_price = high
                hi_extreme_idx = current_idx
                first_high = True
                first_low = True
                line_value = LINE_VALUE_DOWN
                if NoRepaint:
                    draw(lineType, low_extreme_idx, LINE_VALUE_DOWN)
                drawStableLine(lineType, low_extreme_idx, LINE_VALUE_FLAT)

    # Check for Top
    if extreme_mode < 1:
        if high > hi_extreme_price:
            if not first_high:
                eraseExtreme(lineType, hi_extreme_idx, LINE_VALUE_UP)
            hi_extreme_price = high
            hi_extreme_idx = current_idx
            first_high = False
        elif high < hi_extreme_price:
            drawExtreme(lineType, hi_extreme_idx, LINE_VALUE_UP)
            first_high = False
            if ((hi_extreme_price - low) >= min_extreme_height) and ((current_idx - hi_extreme_idx) >= min_extreme_width):
                extreme_mode = 1
                low_extreme_price = low
                low_extreme_idx = current_idx
                first_high = True
                first_low = True
                line_value = LINE_VALUE_UP
                if NoRepaint:
                    draw(lineType, hi_extreme_idx, LINE_VALUE_UP)
                drawStableLine(lineType, hi_extreme_idx, LINE_VALUE_FLAT)

    return low_extreme_idx, hi_extreme_idx, low_extreme_price, hi_extreme_price, first_low, first_high, extreme_mode, signal

# Process each row
for idx in range(1, len(df)):
    for lineType in ['minor', 'major']:
        min_extreme_height = df[f'{lineType.capitalize()}MinExtremeHeight'].iloc[idx]
        state = state_vars[lineType]
        
        state['low_extreme_idx'], state['hi_extreme_idx'], state['low_extreme_price'], state['hi_extreme_price'], state['first_low'], state['first_high'], state['extreme_mode'], _ = check_for_extremes(
            state['low_extreme_idx'], state['low_extreme_price'],
            state['hi_extreme_idx'], state['hi_extreme_price'], state['first_low'], state['first_high'], state['extreme_mode'],
            min_extreme_height, MINOR_MIN_EXTREME_WIDTH if lineType == 'minor' else MAJOR_MIN_EXTREME_WIDTH,
            idx, df['low'].iloc[idx], df['high'].iloc[idx], lineType
        )

# Save the result to a new CSV
df.to_csv("common/MachineLearningModel/output/test_result_9.csv", index=False)
